<a href="https://colab.research.google.com/github/SebGalvaleira/mineria-datos-ii-parcial/blob/main/notebooks/e2/E2_02_bronze_streaming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E2 · 02 · Streaming de eventos → Bronze
Ingesta de `usage_events_stream/*.jsonl` con Structured Streaming: esquema explícito, checkpoint, watermark
sobre el tiempo de llegada, deduplicación por `event_id` en tres barreras y prueba de recuperación ante un fallo.

In [ ]:
!pip install -q "pyspark>=4.0,<4.1"
import os, shutil, subprocess
os.environ.pop('JAVA_HOME', None)

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
LAKE_DRIVE = Path('/content/drive/MyDrive/cpa_datalake')   # respaldo persistente
LAKE       = Path('/content/cpa_datalake')                  # copia de trabajo (disco local de Colab)

RUTAS = {z: LAKE / z for z in ['landing', 'stream_input', 'bronze', 'silver',
                               'gold', 'quarantine', 'checkpoints', 'logs']}

def restaurar(*rutas):
    """Copia desde Drive SOLO las carpetas que este notebook necesita (si no están ya en local)."""
    for r in rutas:
        origen, destino = LAKE_DRIVE / r, LAKE / r
        if origen.exists() and not destino.exists():
            shutil.copytree(origen, destino)
            print('✓ restaurado:', r)

def guardar_en_drive(*rutas):
    """Sincroniza en Drive SOLO las carpetas indicadas."""
    for r in rutas:
        origen, destino = LAKE / r, LAKE_DRIVE / r
        destino.mkdir(parents=True, exist_ok=True)
        subprocess.run(['rsync', '-a', '--delete', f'{origen}/', f'{destino}/'], check=True)
        print('✓ sincronizado:', r)

from pyspark.sql import SparkSession, functions as F, types as T
spark = (SparkSession.builder.appName('cloud-provider-analytics')
         .config('spark.sql.session.timeZone', 'UTC')
         .config('spark.sql.shuffle.partitions', '8')
         .config('spark.sql.sources.partitionOverwriteMode', 'dynamic')
         .getOrCreate())
print('Spark', spark.version)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Spark 4.0.5


In [ ]:
# restaurar solo lo que este notebook necesita
restaurar('landing', 'stream_input', 'bronze/usage_events', 'checkpoints', 'quarantine', 'logs')
for p in RUTAS.values():
    p.mkdir(parents=True, exist_ok=True)

In [ ]:
#Esquema de envtos
S, D = T.StringType(), T.DoubleType()
campo = lambda nombre, tipo: T.StructField(nombre, tipo, True)

ESQUEMA_EVENTOS = T.StructType([
    campo('event_id', S), campo('timestamp', T.TimestampType()), campo('org_id', S),
    campo('resource_id', S), campo('service', S), campo('region', S), campo('metric', S),
    campo('value', S),                       # llega como número o como texto: Bronze lo guarda tal cual
    campo('unit', S), campo('cost_usd_increment', D), campo('schema_version', T.IntegerType()),
    campo('carbon_kg', D),                   # solo v2
    campo('genai_tokens', T.LongType()),     # solo v2 y servicio genai
    campo('_corrupt_record', S)])

ORIGEN_EVENTOS = RUTAS['landing'] / 'usage_events_stream'
BRONZE_EV      = RUTAS['bronze'] / 'usage_events'
CHECKPOINT     = RUTAS['checkpoints'] / 'bronze_usage_events'

In [ ]:
import shutil
# Borra la simulación anterior: llegadas, Bronze de eventos y checkpoint. NO toca Landing.
for p in [RUTAS['stream_input'], BRONZE_EV, CHECKPOINT, RUTAS['quarantine'] / 'usage_events']:
    shutil.rmtree(p, ignore_errors=True)
RUTAS['stream_input'].mkdir(parents=True, exist_ok=True)
print('Simulación reiniciada')

(RUTAS['logs'] / 'stream_bronze.jsonl').unlink(missing_ok=True)

Simulación reiniciada


In [ ]:
#simula la llegada de archivos
def llegar(cantidad):
    """Copia los próximos `cantidad` archivos de Landing a stream_input, como si llegaran ahora."""
    ya = {p.name for p in RUTAS['stream_input'].glob('*.jsonl')}
    pendientes = [p for p in sorted(ORIGEN_EVENTOS.glob('*.jsonl')) if p.name not in ya]
    for p in pendientes[:cantidad]:
        shutil.copy(p, RUTAS['stream_input'] / p.name)
    total = len(list(RUTAS['stream_input'].glob('*.jsonl')))
    print(f'Llegaron {min(cantidad, len(pendientes))} archivos · total en stream_input: {total}/120')

In [ ]:
#el procesamiento de cada micro-lote
import json
FALLA = {'en_batch': None}       # para simular una caída en un lote concreto

def procesar_lote(df, batch_id):
    df = df.persist()
    invalidos = df.filter(F.col('_corrupt_record').isNotNull() | F.col('event_id').isNull())
    validos   = (df.filter(F.col('_corrupt_record').isNull() & F.col('event_id').isNotNull())
                   .drop('_corrupt_record', 'dedupe_key'))

    # Barrera 3: anti-join contra todo lo que ya está en Bronze
    if BRONZE_EV.exists() and any(BRONZE_EV.glob('event_date=*')):
        ya_en_bronze = spark.read.parquet(str(BRONZE_EV)).select('event_id')
        nuevos = validos.join(ya_en_bronze, 'event_id', 'left_anti')
    else:
        nuevos = validos
    nuevos = nuevos.withColumn('stream_batch_id', F.lit(batch_id)).persist()

    n = {'batch_id': batch_id, 'entrada': df.count(), 'invalidos': invalidos.count(),
         'validos': validos.count(), 'nuevos': nuevos.count()}
    n['ya_existian'] = n['validos'] - n['nuevos']

    nuevos.write.mode('append').partitionBy('event_date').parquet(str(BRONZE_EV))
    if n['invalidos']:
        (invalidos.withColumn('motivo', F.lit('parse_error_o_sin_event_id'))
         .write.mode('append').parquet(str(RUTAS['quarantine'] / 'usage_events')))

    # Simulación de una caída DESPUÉS de escribir y ANTES de que Spark confirme el lote
    if FALLA['en_batch'] == batch_id:
        FALLA['en_batch'] = None
        raise RuntimeError(f'Falla simulada en el lote {batch_id} (después de escribir en Bronze)')

    with open(RUTAS['logs'] / 'stream_bronze.jsonl', 'a') as f:
        f.write(json.dumps(n) + '\n')
    print(n)
    nuevos.unpersist(); df.unpersist()

In [ ]:
#la query del streaming

def correr_stream():
    stream = (spark.readStream
              .schema(ESQUEMA_EVENTOS)
              .option('mode', 'PERMISSIVE')
              .option('columnNameOfCorruptRecord', '_corrupt_record')
              .option('maxFilesPerTrigger', 10)
              .json(str(RUTAS['stream_input']))
              .withColumn('source_file', F.col('_metadata.file_path'))                    # linaje
              .withColumn('ingest_ts',   F.current_timestamp())                           # tiempo de LLEGADA
              .withColumn('event_date',  F.to_date('timestamp'))                          # partición por tiempo de EVENTO
              .withColumn('dedupe_key',  F.coalesce('event_id', F.expr('uuid()')))        # filas sin id nunca se fusionan
              .withWatermark('ingest_ts', '1 hour')                                       # sobre la llegada, no sobre el evento
              .dropDuplicatesWithinWatermark(['dedupe_key']))                             # Barrera 2
    q = (stream.writeStream
         .foreachBatch(procesar_lote)
         .option('checkpointLocation', str(CHECKPOINT))                                   # Barrera 1
         .trigger(availableNow=True)
         .start())
    try:
        q.awaitTermination()
        print('Stream terminado sin errores')
    except Exception as e:
        print(' El stream se detuvo:', str(e).splitlines()[0][:200])

def estado_bronze():
    df = spark.read.parquet(str(BRONZE_EV))
    return {'filas': df.count(), 'event_id_distintos': df.select('event_id').distinct().count(),
            'dias': df.select('event_date').distinct().count()}

In [ ]:
#primera tanda (40 archivos
llegar(40)
correr_stream()
estado_bronze()

Llegaron 40 archivos · total en stream_input: 40/120
{'batch_id': 0, 'entrada': 3600, 'invalidos': 0, 'validos': 3600, 'nuevos': 3600, 'ya_existian': 0}
{'batch_id': 1, 'entrada': 3600, 'invalidos': 0, 'validos': 3600, 'nuevos': 3600, 'ya_existian': 0}
{'batch_id': 2, 'entrada': 3600, 'invalidos': 0, 'validos': 3600, 'nuevos': 3600, 'ya_existian': 0}
{'batch_id': 3, 'entrada': 3600, 'invalidos': 0, 'validos': 3600, 'nuevos': 3600, 'ya_existian': 0}
{'batch_id': 4, 'entrada': 0, 'invalidos': 0, 'validos': 0, 'nuevos': 0, 'ya_existian': 0}
Stream terminado sin errores


{'filas': 14400, 'event_id_distintos': 14400, 'dias': 60}

In [ ]:
#2da tanda con falla simulada
llegar(40)
FALLA['en_batch'] = 5            # el 2.º lote de esta tanda se "cae" después de escribir
correr_stream()
print('Después de la caída:', estado_bronze())

Llegaron 40 archivos · total en stream_input: 80/120


ERROR:py4j.clientserver:There was an exception while executing the Python Proxy on the Python Side.
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/py4j/clientserver.py", line 644, in _call_proxy
    return_value = getattr(self.pool[obj_id], method)(*params)
  File "/usr/local/lib/python3.13/dist-packages/pyspark/sql/utils.py", line 165, in call
    raise e
  File "/usr/local/lib/python3.13/dist-packages/pyspark/sql/utils.py", line 162, in call
    self.func(DataFrame(jdf, wrapped_session_jdf), batch_id)
    ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_18531/267678241.py", line 31, in procesar_lote
    raise RuntimeError(f'Falla simulada en el lote {batch_id} (después de escribir en Bronze)')
RuntimeError: Falla simulada en el lote 5 (después de escribir en Bronze)


 El stream se detuvo: [STREAM_FAILED] Query [id = 23455427-7055-4b00-8174-ab73dbd576b2, runId = 25d895a1-13cb-4754-821f-99b35337a599] terminated with exception: [FOREACH_BATCH_USER_FUNCTION_ERROR] An error occurred in the 
Después de la caída: {'filas': 18000, 'event_id_distintos': 18000, 'dias': 60}


In [ ]:
#recuperacion (se reinicia con el mismo checkpoint)
correr_stream()
print('Después de recuperar:', estado_bronze())

{'batch_id': 5, 'entrada': 3600, 'invalidos': 0, 'validos': 3600, 'nuevos': 0, 'ya_existian': 3600}
{'batch_id': 6, 'entrada': 3600, 'invalidos': 0, 'validos': 3600, 'nuevos': 3600, 'ya_existian': 0}
{'batch_id': 7, 'entrada': 3600, 'invalidos': 0, 'validos': 3600, 'nuevos': 3600, 'ya_existian': 0}
{'batch_id': 8, 'entrada': 3600, 'invalidos': 0, 'validos': 3600, 'nuevos': 3600, 'ya_existian': 0}
{'batch_id': 9, 'entrada': 0, 'invalidos': 0, 'validos': 0, 'nuevos': 0, 'ya_existian': 0}
Stream terminado sin errores
Después de recuperar: {'filas': 28800, 'event_id_distintos': 28800, 'dias': 60}


In [ ]:
#ultima tanda y verificacion final
llegar(40)
correr_stream()
final = estado_bronze()
print(final)
assert final['filas'] == final['event_id_distintos'] == 43_200, 'Hay duplicados o faltan eventos'
print('43.200 eventos, todos únicos, en', final['dias'], 'particiones por día')

import pandas as pd
display(pd.read_json(RUTAS['logs'] / 'stream_bronze.jsonl', lines=True))
!du -sh "{BRONZE_EV}"

Llegaron 40 archivos · total en stream_input: 120/120
{'batch_id': 10, 'entrada': 3600, 'invalidos': 0, 'validos': 3600, 'nuevos': 3600, 'ya_existian': 0}
{'batch_id': 11, 'entrada': 3600, 'invalidos': 0, 'validos': 3600, 'nuevos': 3600, 'ya_existian': 0}
{'batch_id': 12, 'entrada': 3600, 'invalidos': 0, 'validos': 3600, 'nuevos': 3600, 'ya_existian': 0}
{'batch_id': 13, 'entrada': 3600, 'invalidos': 0, 'validos': 3600, 'nuevos': 3600, 'ya_existian': 0}
{'batch_id': 14, 'entrada': 0, 'invalidos': 0, 'validos': 0, 'nuevos': 0, 'ya_existian': 0}
Stream terminado sin errores
{'filas': 43200, 'event_id_distintos': 43200, 'dias': 60}
43.200 eventos, todos únicos, en 60 particiones por día


,batch_id,entrada,invalidos,validos,nuevos,ya_existian
0,0,3600,0,3600,3600,0
1,1,3600,0,3600,3600,0
2,2,3600,0,3600,3600,0
3,3,3600,0,3600,3600,0
4,4,0,0,0,0,0
5,5,3600,0,3600,0,3600
6,6,3600,0,3600,3600,0
7,7,3600,0,3600,3600,0
8,8,3600,0,3600,3600,0
9,9,0,0,0,0,0


70M	/content/cpa_datalake/bronze/usage_events


In [ ]:
guardar_en_drive('stream_input', 'bronze/usage_events', 'checkpoints', 'quarantine', 'logs')

Data Lake sincronizado en Drive


## cierre - streaming -> bronze

**que se hizo**
-ingesta de los 120 microlotes de `usage_events_stream/*.jsonl` con **Structured Streaming** (`trigger(availableNow)`, `maxFilesPerTrigger = 10`), simulando su llegada en 3 tandas de 40 archivos.
- **esquema explicito v2**: en los eventos v1, `carbon_kg` y `genai_tokens` quedan nulos. `value` se guarda como texto (Bronze no transforma).
- **Linaje:** `source_file`, `ingest_ts` (llegada) y `stream_batch_id`. **Partición** por `event_date` (tiempo de evento).
- **Tres barreras contra duplicados:**

1. **checkpoint:** cada archivo se lee una sola vez.
2. **`dropDuplicatesWithinWatermark(event_id)`** con watermarrk de 1 h sobre `ingest_ts`: atrapa duplicados cercanos y acota el estado (los lotes sin datos 4, 9 y 14 son los que avanzan el watermark y limpian ese estado).
3. **anti-join contra bronze** en `foreachBatch`: garantia final ante duplicados tardios y reprocesos.
-el watermark va sobre el **tiempo de llegada** y no sobre el del evento: sobre el evento descartaria el 99% de los datos (entrega 1, notebook 04)

**prueba: micro-lote reprocesado  despues de un fallo** (feedback de la entrega 1)
| Momento | Filas en Bronze | event_id distintos |
|---|---|---|
| Tras la tanda 1 | 14.400 | 14.400 |
| Caída simulada en el lote 5 (después de escribir, antes de confirmar) | 18.000 | 18.000 |
| Recuperación: el lote 5 se reprocesa → `nuevos = 0`, `ya_existian = 3.600` | 28.800 | 28.800 |
| Final (120 archivos) | **43.200** | **43.200** |

Spark ofrece *at-least-once* en `foreachBatch`: un lote no confirmado se reprocesa. La idempotencia la garantiza la barrera 3, no el checkpoint.

**Hallazgos**
- **Consistencia eventual de Google Drive:** al leer Bronze inmediatamente después de escribirlo, Drive devolvía Parquet de 0 bytes. Decisión: el pipeline trabaja en el disco local de Colab y sincroniza con Drive al final (`guardar_en_drive`).
- **Archivos chicos:** cada lote escribe en las 60 particiones de fecha, porque los eventos llegan desordenados; Bronze ocupa ~70 MB frente a ~13 MB de JSON. Se resuelve con compactación en el cierre diario (diseño §7.2).

**Evidencia:** `logs/stream_bronze.jsonl` (un registro por lote) y los conteos de esta sección.
